# SpatioLD Full SlideTag Pipeline Notebook

This notebook is a full end-to-end workflow using the **SpatioLD package** and the SlideTag example metadata.

It covers:
- real data loading (metadata + real expression CSV)
- local diversity + permutation inference
- summaries and spatial visualizations
- clustering on local diversity profiles
- gene-radius modeling (updated SlideTag-style functions)
- SVG/HVG comparison and optional Venn visualization

All major steps are split into separate sections/cells for readability.


## 0) Prerequisites

Expected environment:
- `spatiold` installed (`pip install -e .` from `SpatioLD/`)
- core deps: numpy, pandas, scipy, scikit-learn, matplotlib, seaborn, statsmodels, anndata
- optional: scanpy (`pip install -e .[omics]`), matplotlib-venn (`pip install -e .[plot]`)

This notebook tries to degrade gracefully for optional parts.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

import spatiold as sld

sns.set_context("notebook")
sns.set_style("whitegrid")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 1) Configuration

In [2]:
# -----------------------------
# Runtime controls
# -----------------------------
RADI = [30, 60, 90, 120, 150, 180, 210, 240, 270]
N_PERM_PVAL = 100
N_PERM_DIST = 100
MAX_CELLS = None           # set e.g. 5000 for faster iteration
N_MODEL_GENES = None       # genes used for gene-level modeling/SVG
RNG_SEED = 42

# -----------------------------
# Robust path resolution
# -----------------------------
def find_file(candidates):
    for p in candidates:
        path = Path(p).expanduser().resolve()
        if path.exists():
            return path
    raise FileNotFoundError(
        "None of the candidate paths exists:\n" + "\n".join(str(c) for c in candidates)
    )

cwd = Path.cwd().resolve()
parents = [cwd] + list(cwd.parents)

metadata_candidates = []
expr_candidates = ['/Users/zhaotianxiao/Library/CloudStorage/Dropbox/FenyoLab/Project/SpatialStat/data/SlideTag_Human_Brain_SCP2167/humancortex_expression.csv']
for base in parents:
    metadata_candidates += [
        base / "SpatioLD" / "example_data" / "SlideTag_HumanCortex.csv",
        base / "example_data" / "SlideTag_HumanCortex.csv",
    ]
    expr_candidates += [
        base / "data" / "SlideTag_Human_Brain_SCP2167" / "humancortex_expression.csv",
        base / "../data" / "SlideTag_Human_Brain_SCP2167" / "humancortex_expression.csv",
    ]

metadata_path = find_file(metadata_candidates)
expr_path = find_file(expr_candidates)

print("Metadata:", metadata_path)
print("Expression:", expr_path)


Metadata: /Users/zhaotianxiao/Library/CloudStorage/Dropbox/FenyoLab/Project/SpatialStat/SpatioLD/example_data/SlideTag_HumanCortex.csv
Expression: /Users/zhaotianxiao/Library/CloudStorage/Dropbox/FenyoLab/Project/SpatialStat/data/SlideTag_Human_Brain_SCP2167/humancortex_expression.csv


## 2) Load Real Metadata

In [ ]:
meta = pd.read_csv(metadata_path)
if "unique_id" in meta.columns:
    meta = meta.set_index("unique_id")

required_cols = {"x", "y", "cell_type"}
missing = required_cols.difference(meta.columns)
if missing:
    raise ValueError(f"Metadata missing required columns: {sorted(missing)}")

meta = meta.dropna(subset=["x", "y", "cell_type"]).copy()
meta.index = meta.index.astype(str)

if MAX_CELLS is not None and len(meta) > MAX_CELLS:
    rng = np.random.default_rng(RNG_SEED)
    picked = rng.choice(meta.index.to_numpy(), size=MAX_CELLS, replace=False)
    meta = meta.loc[picked].copy()

print("Metadata shape:", meta.shape)
print("Unique cell types:", meta["cell_type"].astype(str).nunique())
meta["cell_type"].astype(str).value_counts().head(10)


## 3) Load Real Expression Data

In [4]:
expr_raw = pd.read_csv(expr_path, index_col=0)
meta_ids = set(meta.index)

# Determine orientation automatically
overlap_index = len(set(expr_raw.index.astype(str)).intersection(meta_ids))
overlap_cols = len(set(map(str, expr_raw.columns)).intersection(meta_ids))

if overlap_cols > overlap_index:
    expr = expr_raw.T
else:
    expr = expr_raw

expr.index = expr.index.astype(str)
expr.columns = expr.columns.astype(str)

print("Expression raw shape:", expr_raw.shape)
print("Expression oriented shape (cells x genes):", expr.shape)
print("Overlap with metadata IDs:", len(expr.index.intersection(meta.index)))


Expression raw shape: (36601, 17441)
Expression oriented shape (cells x genes): (17441, 36601)
Overlap with metadata IDs: 17441


## 4) Expression Preprocessing + Alignment

In [ ]:
expr_filt = sld.preprocess_expression_matrix(
    expr,
    min_fraction_expressed=0.02,
    min_genes_per_cell=50,
)

expr_aligned, meta_aligned = sld.align_expression_and_metadata(expr_filt, meta)

print("Filtered expression shape:", expr_filt.shape)
print("Aligned expression shape:", expr_aligned.shape)
print("Aligned metadata shape:", meta_aligned.shape)


## 5) AnnData-Compatible SpatioLD Object

In [ ]:
adata = ad.AnnData(
    X=expr_aligned.values,
    obs=meta_aligned.copy(),
    var=pd.DataFrame(index=expr_aligned.columns.astype(str)),
)
adata.obs_names = expr_aligned.index.astype(str)

obj = sld.SpatioLD.from_anndata(
    adata,
    label_key="cell_type",
    coord_keys=("x", "y"),
)

print(obj)
print("AnnData shape:", obj.to_anndata().shape)


## 6) Local Diversity (Functional + Object API)

In [ ]:
ld_df = obj.compute_local_diversity(radii=RADI, key="ld_full")

print("LD shape:", ld_df.shape)
ld_df.head()


## 7) Permutation Inference

In [ ]:
pvals_df = obj.compute_permutation_pvals(
    n_perm=N_PERM_PVAL,
    radii=RADI,
    random_state=RNG_SEED,
    n_jobs=1,
    key="ld_pvals",
)

perm_mean_df = obj.compute_permutation_mean(
    n_perm=N_PERM_PVAL,
    radii=RADI,
    random_state=RNG_SEED,
    n_jobs=1,
    key="ld_perm_mean",
)

perm_dist = obj.compute_permutation_distribution(
    n_perm=N_PERM_DIST,
    radii=RADI,
    random_state=RNG_SEED,
    n_jobs=1,
)

print("pvals shape:", pvals_df.shape)
print("perm mean shape:", perm_mean_df.shape)
print("perm dist shape:", perm_dist.shape)


## 8) Summaries

In [ ]:
entropy_global = obj.compute_global_shannon_entropy()

summary_ct = obj.summarize_local_diversity_by_cell_type(
    local_diversity_key="ld_full",
)

summary_null = obj.compute_sample_vs_null_summary(
    perm_dist,
    local_diversity_key="ld_full",
)

print("Global Shannon entropy:", round(entropy_global, 4))
summary_ct.head(), summary_null.head()


## 9) Visualizations: Core Spatial + Diversity Curves

In [ ]:
ax = sld.plot_spatial_cell_types(obj.metadata)
plt.show()

ax = sld.plot_mean_diversity_by_cell_type(
    summary_ct,
    title="Normalized Mean Local Diversity vs Radius by Cell Type",
    ylabel="Mean Local Diversity (normalized)",
)
plt.show()

ax = sld.plot_sample_vs_null_curve(summary_null)
plt.show()


## 10) KMeans on Local Diversity Profiles

In [ ]:
cluster_labels_df, kmeans_models = obj.cluster_local_diversity_profiles(
    local_diversity_key="ld_full",
    k_values=(2, 3, 4, 5),
)
print(cluster_labels_df.head())

fig, axes = sld.plot_kmeans_spatial_maps(obj.metadata, cluster_labels_df, k_values=[2, 3, 4, 5], ncols=2)
plt.show()


## 11) Significant Local Diversity Maps

In [ ]:
sig_mask_df = obj.build_significance_mask(pvals_key="ld_pvals", alpha=0.05)
print(sig_mask_df.mean().rename("fraction_significant_by_radius"))

fig, axes = sld.plot_significant_diversity_maps(obj.get_coords_df(), pvals_df, alpha=0.05, ncols=3)
plt.show()


## 12) Prepare Genes for Modeling

In [ ]:
# choose top variable genes for runtime practicality
expr_for_model = obj.to_anndata().to_df()
gene_var = expr_for_model.var(axis=0).sort_values(ascending=False)

if N_MODEL_GENES is None:
    model_genes = gene_var.index.tolist()
else:
    model_genes = gene_var.head(int(N_MODEL_GENES)).index.tolist()

expr_model = expr_for_model.loc[:, model_genes].copy()

print("Model expression shape:", expr_model.shape)
expr_model.iloc[:5, :5]


## 13) Updated SlideTag-Style Radius Modeling

In [ ]:
shared = obj.prepare_shared_components(
    local_diversity_key="ld_full",
    radius_mode="poly",   # use "spline" for spline basis
    poly_degree=3,
)

results_df, fit_objects = sld.fit_all_genes(
    expr_model,
    shared,
    cluster_robust=True,
    verbose=True,
)

print("Results rows:", len(results_df))
results_df.head(10)


## 14) Volcano Plot of Gene Effects

In [ ]:
ax = sld.plot_gene_effect_volcano(results_df)
plt.show()


## 15) Reconstruct Radius Effect for Top Gene

In [ ]:
top_gene = results_df.iloc[0]["gene"]
fit_top = fit_objects[top_gene]
f_r = sld.reconstruct_radius_effect(fit_top, shared, include_intercept=True)

plt.figure(figsize=(6, 4), dpi=160)
plt.plot(RADI, f_r, marker="o")
plt.xlabel("Radius")
plt.ylabel("Reconstructed f(r)")
plt.title(f"Reconstructed Radius Effect: {top_gene}")
plt.grid(True, linestyle="--", alpha=0.35)
plt.show()


## 16) Fit a Single Gene Model Explicitly

In [ ]:
example_gene = results_df.iloc[1]["gene"] if len(results_df) > 1 else results_df.iloc[0]["gene"]
one_fit = sld.fit_single_gene_radius_model(expr_model[example_gene].values, shared, cluster_robust=True)

print("Gene:", example_gene)
print(one_fit["coef"].head(10))


## 17) Spatially Variable Genes (Moran's I)

In [ ]:
svg_df = obj.compute_svg_morans_i(expr_model, k=15)
svg_df.head(10)


## 18) HVG + Venn Comparison

In [ ]:
hvg_df = None
try:
    hvg_df = sld.compute_hvg_scanpy(expr_model, n_top_genes=min(100, expr_model.shape[1]))
    print("HVG computed:", len(hvg_df))
except Exception as e:
    print("Skipping HVG step (scanpy unavailable or failed):", e)

if hvg_df is not None:
    hvg_set = set(hvg_df.head(100).index)
    svg_set = set(svg_df.head(100)["gene"])
    ldvg_set = set(results_df.nsmallest(100, "pval_gene")["gene"])

    try:
        ax = sld.plot_gene_set_venn(
            sorted(hvg_set),
            sorted(svg_set),
            sorted(ldvg_set),
            labels=("HVG", "SVG", "LDVG"),
        )
        plt.show()
    except Exception as e:
        print("Skipping Venn plot (matplotlib-venn unavailable or failed):", e)


## 19) Optional: Save Outputs

In [ ]:
# Uncomment to persist outputs
# out_dir = Path("./spatiold_notebook_outputs")
# out_dir.mkdir(parents=True, exist_ok=True)
# ld_df.to_csv(out_dir / "local_diversity.csv")
# pvals_df.to_csv(out_dir / "local_diversity_pvals.csv")
# results_df.to_csv(out_dir / "gene_radius_model_results.csv", index=False)
# svg_df.to_csv(out_dir / "svg_morans_i.csv", index=False)
# summary_ct.to_csv(out_dir / "summary_by_cell_type.csv", index=False)
# summary_null.to_csv(out_dir / "summary_sample_vs_null.csv", index=False)


## 20) Summary Checklist

- [x] Loaded real SlideTag metadata
- [x] Loaded real expression matrix
- [x] Preprocessed and aligned expression + metadata
- [x] Computed local diversity and permutation inference
- [x] Generated core diversity/spatial visualizations
- [x] Ran KMeans on local diversity profiles
- [x] Built significant-local-diversity maps
- [x] Fitted updated per-gene radius models
- [x] Generated volcano plot and reconstructed radius effect
- [x] Computed SVG ranking via Moran's I
- [x] Optional HVG and Venn overlap analysis
